In [ ]:

# ============================================================
# ASSIGNMENT 2: PANDAS DATA CLEANING AND EDA
# ============================================================

import pandas as pd
import numpy as np

# STEP 1: LOAD AND EXPLORE THE DATASET

df = pd.read_csv("sales.csv")

print("FIRST 5 ROWS")
print(df.head())

print("\nDATASET INFORMATION")
df.info()

print("\nSUMMARY STATISTICS")
print(df.describe(include="all"))

print("\nDATASET SHAPE")
print(df.shape)

print("\nCOLUMN NAMES")
print(df.columns.tolist())


# STEP 2: IDENTIFY DATA QUALITY ISSUES

print("\n1. MISSING VALUES")
print(df.isnull().sum())

print("\n2. DUPLICATE ROWS")
print(df.duplicated().sum())

print("\n3. DATA TYPES")
print(df.dtypes)

print("\n4. UNIQUE VALUES IN CATEGORICAL COLUMNS")
for col in df.select_dtypes(include="object").columns:
    print(col, ":", df[col].dropna().unique()[:20])

print("\n5. EXTRA SPACES IN TEXT COLUMNS")
for col in df.select_dtypes(include="object").columns:
    spaces = df[col].dropna().astype(str).str.contains(
        r"^\s|\s$", regex=True
    ).sum()
    print(col, ":", spaces)

print("\n6. POTENTIAL NUMERIC OUTLIERS")
for col in df.select_dtypes(include=np.number).columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((df[col] < lower) | (df[col] > upper)).sum()
    print(col, ":", count)


# STEP 3: CLEAN THE DATASET

clean_df = df.copy()
cleaning_log = []

# A. Remove duplicate rows
before = len(clean_df)
clean_df = clean_df.drop_duplicates()
removed = before - len(clean_df)

cleaning_log.append(
    f"Removed {removed} duplicate rows"
)

# B. Remove extra spaces and standardize text
for col in clean_df.select_dtypes(include="object").columns:
    clean_df[col] = clean_df[col].str.strip()

cleaning_log.append(
    "Removed leading and trailing spaces"
)

# C. Standardize common category values
for col in clean_df.select_dtypes(include="object").columns:
    if col.lower() in ["gender", "sex"]:
        clean_df[col] = clean_df[col].str.lower()
        clean_df[col] = clean_df[col].replace({
            "m": "male",
            "f": "female"
        })

cleaning_log.append(
    "Standardized gender values where applicable"
)

# D. Convert date columns if present
for col in clean_df.columns:
    if "date" in col.lower():
        converted = pd.to_datetime(
            clean_df[col], errors="coerce"
        )

        if converted.notna().sum() > 0:
            clean_df[col] = converted

cleaning_log.append(
    "Converted recognizable date columns"
)

# E. Convert numeric-looking text columns
for col in clean_df.select_dtypes(include="object").columns:
    if any(word in col.lower() for word in [
        "sales", "price", "quantity", "amount", "age",
        "score", "attendance", "fare", "hours"
    ]):
        converted = pd.to_numeric(
            clean_df[col].astype("string").str.replace(
                ",", "", regex=False
            ),
            errors="coerce"
        )

        if converted.notna().sum() > 0:
            clean_df[col] = converted

cleaning_log.append(
    "Converted suitable numeric columns"
)

# F. Fill missing values
for col in clean_df.columns:
    missing = clean_df[col].isna().sum()

    if missing > 0:
        if pd.api.types.is_numeric_dtype(clean_df[col]):
            median_value = clean_df[col].median()

            if pd.notna(median_value):
                clean_df[col] = clean_df[col].fillna(
                    median_value
                )
                cleaning_log.append(
                    f"Filled missing {col} values with median"
                )

        else:
            mode_values = clean_df[col].mode(dropna=True)

            if not mode_values.empty:
                clean_df[col] = clean_df[col].fillna(
                    mode_values.iloc[0]
                )
                cleaning_log.append(
                    f"Filled missing {col} values with mode"
                )

print("\nCLEANING LOG")
for change in cleaning_log:
    print("-", change)

print("\nMISSING VALUES AFTER CLEANING")
print(clean_df.isnull().sum())

print("\nDUPLICATES AFTER CLEANING")
print(clean_df.duplicated().sum())


# OPTIONAL: CHECK OUTLIERS USING IQR

print("\nOUTLIER CHECK AFTER CLEANING")

for col in clean_df.select_dtypes(include=np.number).columns:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((clean_df[col] < lower) |
             (clean_df[col] > upper)).sum()

    print(col, "potential outliers:", count)


# STEP 4: GROUPBY ANALYSIS

# Find a suitable region/category column and sales column
region_col = next(
    (c for c in clean_df.columns
     if c.lower() in ["region", "country", "territory"]),
    None
)

sales_col = next(
    (c for c in clean_df.columns
     if c.lower() in ["sales", "total sales", "revenue"]),
    None
)

category_col = next(
    (c for c in clean_df.columns
     if c.lower() in ["category", "product category",
                      "productline", "product line"]),
    None
)

if region_col and sales_col:
    print("\nTOTAL SALES BY REGION")
    print(
        clean_df.groupby(region_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
    )

elif category_col and sales_col:
    print("\nTOTAL SALES BY PRODUCT CATEGORY")
    print(
        clean_df.groupby(category_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
    )

else:
    print(
        "\nFor GroupBy, select a categorical column "
        "and a numeric column from your dataset."
    )


# STEP 5: CREATE A PIVOT TABLE

if region_col and category_col and sales_col:
    print("\nPIVOT TABLE: REGION VS CATEGORY")

    pivot = pd.pivot_table(
        clean_df,
        values=sales_col,
        index=region_col,
        columns=category_col,
        aggfunc="sum",
        fill_value=0
    )

    print(pivot)

elif category_col and sales_col:
    print("\nPIVOT TABLE: PRODUCT CATEGORY")

    pivot = pd.pivot_table(
        clean_df,
        values=sales_col,
        index=category_col,
        aggfunc="sum",
        fill_value=0
    )

    print(pivot)

else:
    print(
        "\nCheck the dataset columns to create "
        "a pivot table using suitable columns."
    )


# STEP 6: DATA QUALITY SUMMARY - EXACTLY FIVE LINES

summary = [
    "1. The dataset was explored using head(), info(), describe(), and shape.",
    "2. Missing values, duplicate records, data types, text consistency, and potential outliers were checked.",
    "3. Duplicate rows were removed and extra spaces in text values were trimmed.",
    "4. Suitable text, date, and numeric columns were standardized, and missing values were handled where possible.",
    "5. The cleaned dataset was checked again and prepared for further analysis."
]

print("\nDATA QUALITY SUMMARY")
for line in summary:
    print(line)


# STEP 7: SAVE THE CLEANED DATASET

clean_df.to_csv("cleaned_data.csv", index=False)

print("\nCleaned dataset saved as cleaned_data.csv")

# Preview the cleaned dataset
print("\nFINAL CLEANED DATA")
print(clean_df.head())
